In [1]:
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GRU, BatchNormalization,LSTM,Input
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler,RobustScaler
import pickle
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,mean_absolute_percentage_error
import os
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
import pandas as pd

df = pd.read_pickle('/content/Copy of df_clean.pkl')
df.head()

,Date,Symbol,Open,High,Low,Close,VWAP,Volume,Deliverable Volume
1023,2012-01-17,ADANIPORTS,137.1,141.00,135.00,140.00,138.13,1636196,1004327.0
1024,2012-01-18,ADANIPORTS,142.0,143.80,138.70,141.70,141.25,890591,404925.0
1025,2012-01-19,ADANIPORTS,144.0,150.55,143.15,149.40,146.72,1456077,721545.0
1026,2012-01-20,ADANIPORTS,151.9,157.60,150.25,155.40,153.76,1634070,861145.0
1027,2012-01-23,ADANIPORTS,155.4,155.40,145.10,146.75,149.54,1657609,820653.0


In [ ]:
len(df['Symbol'].value_counts())

65

In [ ]:
features = [
    'Open', 'High', 'Low', 'Close', 'VWAP', 'Volume',
    'Deliverable Volume',
    'return', 'MA10', 'MA50', 'volatility','MACD', 'High_Low_Ratio', 'Close_Open_Ratio'
]

lookback = 60
train_ratio = 0.8
val_ratio = 0.1
epochs = 100
batch_size = 32
lr = 0.0005
min_rows = 500

os.makedirs('models', exist_ok=True)

excluded_name = {
    'HINDALC0',    # Old symbol for HINDALCO
    'HEROHONDA',   # Old symbol for HEROMOTOCO
    'HINDLEVER',   # Old symbol for HINDUNILVR
    'INFOSYSTCH',  # Old symbol for INFY
    'TISCO',       # Old symbol for TATASTEEL
    'MUNDRAPORT',  # Old symbol for ADANIPORTS
    'SESAGOA',     # Old symbol for VEDL
    'ZEETELE',     # Old symbol for ZEEL
    'UTIBANK',     # Old symbol for AXISBANK
    'SSLT',        # Old symbol for VEDL (Sterlite)
    'KOTAKMAH',    # Old symbol for KOTAKBANK
    'TELCO',       # Old symbol for TATAMOTORS
    'BHARTI',      # Old symbol for BHARTIARTL
    'UNIPHOS',     # Old symbol (UPL predecessor)
}
all_stocks_raw = sorted(df['Symbol'].unique())
all_stocks = [s for s in all_stocks_raw if s not in excluded_name]


preprocessing

In [ ]:
def feature_creation(df,stock_name):
    data = df[df['Symbol'] == stock_name].copy()
    data.sort_values('Date', inplace=True)

    data['return'] = data['Close'].pct_change()
    data['MA10'] = data['Close'].rolling(10).mean()
    data['MA50'] = data['Close'].rolling(50).mean()
    data['volatility'] = data['Close'].rolling(10).std()

    data = data.dropna().reset_index(drop=True)

    ema12 = data['Close'].ewm(span=12, adjust=False).mean()
    ema26 = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = ema12 - ema26

    data['High_Low_Ratio'] = data['High'] / (data['Low'] + 1e-10)  # Avoid division by zero
    data['Close_Open_Ratio'] = data['Close'] / (data['Open'] + 1e-10)  # Avoid division by zero

    data['target'] = data['Close'].pct_change().shift(-1)
    data['current_close'] = data['Close']

    data = data.dropna().reset_index(drop=True)
    return data

In [ ]:
def model_creation(lookback,feature_count,lr=0.0005):
    model = Sequential()

    model.add(Input(shape=(lookback, len(feature_count)))) #Input layer to specify the shape of input data to the model. In this case, it's a sequence of 'lookback' time steps, each with 'len(features)' features.

    model.add(GRU(units=128, return_sequences=True))
    #in gru input shape is (time_Stamp,feature)
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    model.add(GRU(units=64, return_sequences=True))
    #in gru input shape is (time_Stamp,feature)
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    model.add(GRU(units=32, return_sequences=False))
    #in gru input shape is (time_Stamp,feature)
    model.add(BatchNormalization())
    model.add(Dropout(0.2))

    model.add(Dense(16, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1))  # GRU outputs hidden features, not the final prediction.Dense layer converts those features into final output.

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='huber'
    )

    return model

In [ ]:
def train_stock(df,stock_name,features,lookback,train_ratio,val_ratio,epochs,batch_size,lr):

    data = feature_creation(df,stock_name)

    if len(data) < lookback + 50:  # Need minimum data
        print(f" Skipping {stock_name}: only {len(data)} rows (too few)")
        return None

    train_end = int(len(data) * train_ratio)
    val_end = int(len(data) * (train_ratio + val_ratio))

    train_data = data.iloc[:train_end]
    val_data = data.iloc[train_end:val_end]
    test_data = data.iloc[val_end:] # Test data is the last 10% of the data

    scaler_X = RobustScaler()
    scaler_y = RobustScaler()

    x_train_scale = scaler_X.fit_transform(train_data[features])
    x_test_scale = scaler_X.transform(test_data[features])
    x_val_scale = scaler_X.transform(val_data[features])

    y_train_scale = scaler_y.fit_transform(train_data[['target']])
    y_test_scale = scaler_y.transform(test_data[['target']])
    y_val_scale = scaler_y.transform(val_data[['target']])

    def make_sequences(X_scaled, y_scaled, lookback):
        x_all , y_all = [], []
        n = min(len(X_scaled), len(y_scaled)) #for do shifting length will be different so we take min length to avoid index error

        for i in range(lookback, n):
            x_all.append(X_scaled[i-lookback:i])
            y_all.append(y_scaled[i])

        return np.array(x_all), np.array(y_all)

    x_train,y_train = make_sequences(x_train_scale, y_train_scale, lookback)
    x_val,y_val = make_sequences(x_val_scale, y_val_scale, lookback)
    x_test,y_test = make_sequences(x_test_scale, y_test_scale, lookback)

    if len(x_val) < 10 or len(x_test) < 10:
        print(f"Skipping {stock_name}: val ({len(x_val)}) or test ({len(x_test)}) too small")
        return None

    tf.keras.backend.clear_session() #clears the current TensorFlow/Keras ession and releases memory used by previous models.
    model = model_creation(lookback,features)

    history = model.fit(
        x_train,y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(x_val,y_val),
        callbacks = [EarlyStopping(monitor='val_loss', patience=10,restore_best_weights=True,verbose=0),
                    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5,verbose=0,min_lr=1e-6)],
        shuffle=False,
        verbose=0
    )

    pred = scaler_y.inverse_transform(model.predict(x_test, verbose=0)).flatten()
    actual = scaler_y.inverse_transform(y_test).flatten()

    mae_return = mean_absolute_error(actual, pred)


    actual_dir = np.sign(actual)
    pred_dir   = np.sign(pred)
    dir_acc = np.mean(actual_dir == pred_dir) * 100

    # ── CONVERT TO PRICE for secondary metrics ──
    # predicted_price = current_close * (1 + predicted_return)
    test_close = test_data['current_close'].values[lookback:]
    pred_close = test_close * (1 + pred)
    actual_close = test_close * (1 + actual)

    mae_price = mean_absolute_error(actual_close, pred_close)
    mape_price = mean_absolute_percentage_error(actual_close, pred_close) * 100

    model.save(f'models/gru_model_{stock_name}.keras')
    pickle.dump(scaler_X, open(f'models/scaler_X_{stock_name}.pkl', 'wb'))
    pickle.dump(scaler_y, open(f'models/scaler_y_{stock_name}.pkl', 'wb'))

    return {
        'Stock': stock_name,
        'Rows': len(data),
        'Train': len(x_train),
        'Test': len(x_test),
        'MAE': round(mae_return, 2),
        'MAE_Price': round(mae_price, 2),
        'MAPE_Price': round(mape_price, 2),
        'Direction_Accuracy': round(dir_acc, 2)
    }

In [ ]:
results = []

for i,stock in enumerate(all_stocks):
    print(f"\n[{i+1}/{len(all_stocks)}] Training: {stock}...")
    result = train_stock(df,stock,features,lookback,train_ratio,val_ratio,epochs=50,batch_size=32,lr=0.0005)

    if result is not None:
        results.append(result)
        print(f"MAE={result['MAE']:.2f}  MAPE_price={result['MAPE_Price']:.2f} MAE_price={result['MAE_Price']:.2f} diracc = {result['Direction_Accuracy']:.2f}% ")

print(f"\n{'='*60}")
print(f"  DONE! Trained {len(results)} / {len(all_stocks)} models")
print(f"  All saved to 'models/' folder")
print(f"{'='*60}")


[1/51] Training: ADANIPORTS...
MAE=₹0.02  MAPE_price=1.98% MAE_price=10.70% diracc = 49.09% 

[2/51] Training: ASIANPAINT...
MAE=₹0.01  MAPE_price=1.41% MAE_price=27.33% diracc = 47.64% 

[3/51] Training: AXISBANK...
MAE=₹0.03  MAPE_price=2.55% MAE_price=12.66% diracc = 50.55% 

[4/51] Training: BAJAJ-AUTO...
MAE=₹0.01  MAPE_price=1.31% MAE_price=41.86% diracc = 57.81% 

[5/51] Training: BAJAJFINSV...
MAE=₹0.02  MAPE_price=2.05% MAE_price=146.13% diracc = 51.17% 

[6/51] Training: BAJAUTOFIN...
MAE=₹0.02  MAPE_price=1.76% MAE_price=8.02% diracc = 40.61% 

[7/51] Training: BAJFINANCE...
MAE=₹0.02  MAPE_price=2.01% MAE_price=86.85% diracc = 53.03% 

[8/51] Training: BHARTIARTL...
MAE=₹0.02  MAPE_price=1.96% MAE_price=9.94% diracc = 48.68% 

[9/51] Training: BPCL...
MAE=₹0.02  MAPE_price=2.03% MAE_price=8.07% diracc = 50.00% 

[10/51] Training: BRITANNIA...
MAE=₹0.01  MAPE_price=1.24% MAE_price=39.02% diracc = 49.79% 

[11/51] Training: CIPLA...
MAE=₹0.02  MAPE_price=1.54% MAE_price=9.48

In [ ]:
results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

print(f"\n{'='*50}")
print(f"  Average MAPE: {results_df['MAPE_Price'].mean():.2f}%")
print(f"{'='*50}")

     Stock  Rows  Train  Test  MAE  MAE_Price  MAPE_Price  Direction_Accuracy
ADANIPORTS  2249   1739   165 0.02      10.70        1.98               49.09
ASIANPAINT  5256   4144   466 0.01      27.33        1.41               47.64
  AXISBANK  3344   2615   275 0.03      12.66        2.55               50.55
BAJAJ-AUTO  3152   2461   256 0.01      41.86        1.31               57.81
BAJAJFINSV  3151   2460   256 0.02     146.13        2.05               51.17
BAJAUTOFIN  2561   1988   197 0.02       8.02        1.76               40.61
BAJFINANCE  2574   1999   198 0.02      86.85        2.01               53.03
BHARTIARTL  3635   2848   304 0.02       9.94        1.96               48.68
      BPCL  5256   4144   466 0.02       8.07        2.03               50.00
 BRITANNIA  5255   4144   466 0.01      39.02        1.24               49.79
     CIPLA  5256   4144   466 0.02       9.48        1.54               48.50
 COALINDIA  2548   1978   195 0.01       1.95        1.48       

In [63]:
!zip -r models.zip models

  adding: models/ (stored 0%)
  adding: models/gru_model_ADANIPORTS.keras (deflated 12%)
  adding: models/scaler_y_VEDL.pkl (deflated 28%)
  adding: models/scaler_y_NESTLEIND.pkl (deflated 28%)
  adding: models/scaler_X_VEDL.pkl (deflated 23%)
  adding: models/scaler_y_HINDALCO.pkl (deflated 30%)
  adding: models/scaler_y_NTPC.pkl (deflated 30%)
  adding: models/gru_model_RELIANCE.keras (deflated 12%)
  adding: models/gru_model_EICHERMOT.keras (deflated 12%)
  adding: models/gru_model_INFY.keras (deflated 12%)
  adding: models/scaler_X_BAJFINANCE.pkl (deflated 24%)
  adding: models/scaler_y_HCLTECH.pkl (deflated 28%)
  adding: models/scaler_X_M&M.pkl (deflated 24%)
  adding: models/scaler_X_DRREDDY.pkl (deflated 23%)
  adding: models/scaler_y_M&M.pkl (deflated 29%)
  adding: models/scaler_y_COALINDIA.pkl (deflated 30%)
  adding: models/scaler_y_HDFCBANK.pkl (deflated 29%)
  adding: models/scaler_X_MARUTI.pkl (deflated 25%)
  adding: models/scaler_y_JSWSTEEL.pkl (deflated 28%)
  adding:

In [64]:
from google.colab import files
files.download("models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
colors = ['#4CAF50' if r >= 0.8 else '#FF9800' if r >= 0.5 else '#F44336' for r in results_df['R²']]
ax.bar(range(len(results_df)), results_df['R²'], color=colors)
ax.set_xticks(range(len(results_df)))
ax.set_xticklabels(results_df['Stock'], rotation=90, fontsize=8)
ax.set_ylabel('R² Score', fontsize=12)
ax.set_title('R² Score by Stock (Green ≥ 0.8, Orange ≥ 0.5, Red < 0.5)', fontsize=14, fontweight='bold')
ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.5)
ax.axhline(y=0.5, color='orange', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

for lstm

In [2]:
os.makedirs('models_lstm', exist_ok=True)

In [ ]:
def model_creation_lstm(lookback,feature_count,lr=0.0005):
  model1 = Sequential()

  model1.add(LSTM(units=64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))
    #in gru input shape is (time_Stamp,feature)
  model1.add(BatchNormalization())
  model1.add(Dropout(0.3))

  model1.add(LSTM(32))
  model1.add(BatchNormalization())
  model1.add(Dropout(0.3))

  model1.add(Dense(1))  # GRU outputs hidden features, not the final prediction.Dense layer converts those features into final output.

  model1.compile(
      optimizer=Adam(learning_rate=lr),
      loss='mse'
  )

  return model1

In [3]:
def train_stock_lstm(df,stock_name,features,lookback,train_ratio,val_ratio,epochs,batch_size,lr):

    data = feature_creation(df,stock_name)

    if len(data) < lookback + 50:  # Need minimum data
        print(f" Skipping {stock_name}: only {len(data)} rows (too few)")
        return None

    train_end = int(len(data) * train_ratio)
    val_end = int(len(data) * (train_ratio + val_ratio))

    train_data = data.iloc[:train_end]
    val_data = data.iloc[train_end:val_end]
    test_data = data.iloc[val_end:] # Test data is the last 10% of the data

    scaler_X = RobustScaler()
    scaler_y = RobustScaler()

    x_train_scale = scaler_X.fit_transform(train_data[features])
    x_test_scale = scaler_X.transform(test_data[features])
    x_val_scale = scaler_X.transform(val_data[features])

    y_train_scale = scaler_y.fit_transform(train_data[['target']])
    y_test_scale = scaler_y.transform(test_data[['target']])
    y_val_scale = scaler_y.transform(val_data[['target']])

    def make_sequences(X_scaled, y_scaled, lookback):
        x_all , y_all = [], []
        n = min(len(X_scaled), len(y_scaled)) #for do shifting length will be different so we take min length to avoid index error

        for i in range(lookback, n):
            x_all.append(X_scaled[i-lookback:i])
            y_all.append(y_scaled[i])

        return np.array(x_all), np.array(y_all)

    x_train,y_train = make_sequences(x_train_scale, y_train_scale, lookback)
    x_val,y_val = make_sequences(x_val_scale, y_val_scale, lookback)
    x_test,y_test = make_sequences(x_test_scale, y_test_scale, lookback)

    if len(x_val) < 10 or len(x_test) < 10:
        print(f"Skipping {stock_name}: val ({len(x_val)}) or test ({len(x_test)}) too small")
        return None

    tf.keras.backend.clear_session() #clears the current TensorFlow/Keras ession and releases memory used by previous models.
    model = model_creation_lstm(lookback,features)

    history = model.fit(
        x_train,y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(x_val,y_val),
        callbacks = [EarlyStopping(monitor='val_loss', patience=10,restore_best_weights=True,verbose=0),
                    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5,verbose=0,min_lr=1e-6)],
        shuffle=False,
        verbose=0
    )

    pred = scaler_y.inverse_transform(model.predict(x_test, verbose=0)).flatten()
    actual = scaler_y.inverse_transform(y_test).flatten()

    mae_return = mean_absolute_error(actual, pred)


    actual_dir = np.sign(actual)
    pred_dir   = np.sign(pred)
    dir_acc = np.mean(actual_dir == pred_dir) * 100

    # ── CONVERT TO PRICE for secondary metrics ──
    # predicted_price = current_close * (1 + predicted_return)
    test_close = test_data['current_close'].values[lookback:]
    pred_close = test_close * (1 + pred)
    actual_close = test_close * (1 + actual)

    mae_price = mean_absolute_error(actual_close, pred_close)
    mape_price = mean_absolute_percentage_error(actual_close, pred_close) * 100

    model.save(f'models_lstm/lstm_model_{stock_name}.keras')
    pickle.dump(scaler_X, open(f'models_lstm/scaler_X_{stock_name}.pkl', 'wb'))
    pickle.dump(scaler_y, open(f'models_lstm/scaler_y_{stock_name}.pkl', 'wb'))

    return {
        'Stock': stock_name,
        'Rows': len(data),
        'Train': len(x_train),
        'Test': len(x_test),
        'MAE': round(mae_return, 2),
        'MAE_Price': round(mae_price, 2),
        'MAPE_Price': round(mape_price, 2),
        'Direction_Accuracy': round(dir_acc, 2)
    }

In [ ]:
results_of_lstm = []

for i,stock in enumerate(all_stocks):
    print(f"\n[{i+1}/{len(all_stocks)}] Training: {stock}...")
    result = train_stock_lstm(df,stock,features,lookback,train_ratio,val_ratio,epochs=50,batch_size=32,lr=0.0005)

    if result is not None:
        results_of_lstm.append(result)
        print(f"MAE={result['MAE']:.2f}  MAPE_price={result['MAPE_Price']:.2f} MAE_price={result['MAE_Price']:.2f} diracc = {result['Direction_Accuracy']:.2f}% ")

print(f"\n{'='*60}")
print(f"  DONE! Trained {len(results_of_lstm)} / {len(all_stocks)} models")
print(f"  All saved to 'models_lstm/' folder")
print(f"{'='*60}")

In [ ]:
results_df = pd.DataFrame(results_of_lstm)

print(results_df.to_string(index=False))

print(f"\n{'='*50}")
print(f"  Average MAPE: {results_df['MAPE_Price'].mean():.2f}%")
print(f"{'='*50}")